In [6]:
# This cell sets up the notebook environment for the data preparation and EDA phase.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [7]:
orders = pd.read_csv(f'data/olist_orders_dataset.csv')
items = pd.read_csv(f'data/olist_order_items_dataset.csv')
products = pd.read_csv(f'data/olist_products_dataset.csv')
sellers = pd.read_csv(f'data/olist_sellers_dataset.csv')
customers = pd.read_csv(f'data/olist_customers_dataset.csv')
geolocation = pd.read_csv(f'data/olist_geolocation_dataset.csv')

# Keep payments loaded only because later notebook cells still reference it.
# It is not part of the finalized V1 feature set.
payments = pd.read_csv(f'data/olist_order_payments_dataset.csv')

In [ ]:
# This cell removes duplicate rows only from tables that should have one row per key.
unique_key_tables = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'sellers': 'seller_id',
    'products': 'product_id',
}

remove_duplicates_summary = []

# Loop through each table that should have a unique join key.
for table_name, key_col in unique_key_tables.items():
    current_df = globals()[table_name]
    rows_before = len(current_df)
    duplicate_key_rows = current_df.duplicated(subset=key_col).sum()

    # Drop duplicate key rows so later joins do not create accidental duplication.
    current_df = current_df.drop_duplicates(subset=key_col).copy()
    globals()[table_name] = current_df

    remove_duplicates_summary.append({
        'table': table_name,
        'key_checked': key_col,
        'rows_before': rows_before,
        'duplicate_key_rows_found': duplicate_key_rows,
        'rows_after': len(current_df),
    })

pd.DataFrame(remove_duplicates_summary)


,table,key_checked,rows_before,duplicate_key_rows_found,rows_after
0,orders,order_id,99441,0,99441
1,customers,customer_id,99441,0,99441
2,sellers,seller_id,3095,0,3095
3,products,product_id,32951,0,32951


In [ ]:
# This cell checks the row grain of the main tables before we start joining them.
table_grain_check = pd.DataFrame([
    {
        'table': 'orders',
        'expected_row_grain': 'one row per order',
        'key_checked': 'order_id',
        'duplicate_key_count': orders['order_id'].duplicated().sum(),
        'interpretation': 'duplicates would be a problem',
    },
    {
        'table': 'items',
        'expected_row_grain': 'one row per item in an order',
        'key_checked': 'order_id',
        'duplicate_key_count': items['order_id'].duplicated().sum(),
        'interpretation': 'expected because one order can contain multiple items',
    },
    {
        'table': 'customers',
        'expected_row_grain': 'one row per customer_id',
        'key_checked': 'customer_id',
        'duplicate_key_count': customers['customer_id'].duplicated().sum(),
        'interpretation': 'duplicates would be a problem',
    },
    {
        'table': 'sellers',
        'expected_row_grain': 'one row per seller_id',
        'key_checked': 'seller_id',
        'duplicate_key_count': sellers['seller_id'].duplicated().sum(),
        'interpretation': 'duplicates would be a problem',
    },
    {
        'table': 'products',
        'expected_row_grain': 'one row per product_id',
        'key_checked': 'product_id',
        'duplicate_key_count': products['product_id'].duplicated().sum(),
        'interpretation': 'duplicates would be a problem',
    },
    {
        'table': 'payments',
        'expected_row_grain': 'one row per payment record',
        'key_checked': 'order_id',
        'duplicate_key_count': payments['order_id'].duplicated().sum(),
        'interpretation': 'expected because one order can use multiple payment records',
    },
])

# Display the table so we can confirm which repeated keys are expected.
table_grain_check


,table,expected_row_grain,key_checked,duplicate_key_count,interpretation
0,orders,one row per order,order_id,0,duplicates would be a problem
1,items,one row per item in an order,order_id,13984,expected because one order can contain multipl...
2,customers,one row per customer_id,customer_id,0,duplicates would be a problem
3,sellers,one row per seller_id,seller_id,0,duplicates would be a problem
4,products,one row per product_id,product_id,0,duplicates would be a problem
5,payments,one row per payment record,order_id,4446,expected because one order can use multiple pa...


In [ ]:
# This cell summarizes missing values across the main source tables.
source_tables = {
    'orders': orders,
    'items': items,
    'customers': customers,
    'sellers': sellers,
    'products': products,
    'payments': payments,
    'geolocation': geolocation,
}

null_check_rows = []

for table_name, current_df in source_tables.items():
    null_counts = current_df.isna().sum()
    null_counts = null_counts[null_counts > 0]

    # Keep the null details in one row so the result is easier to scan.
    null_check_rows.append({
        'table': table_name,
        'has_nulls': not null_counts.empty,
        'total_nulls': int(null_counts.sum()),
        'null_columns': ', '.join(
            f'{col} ({count})' for col, count in null_counts.items()
        ) if not null_counts.empty else 'none',
    })

null_check_summary = (
    pd.DataFrame(null_check_rows)
    .sort_values(['has_nulls', 'table'], ascending=[False, True])
    .reset_index(drop=True)
)

null_check_summary


In [ ]:
# This cell filters to delivered orders and creates the V1 target and time features.
order_datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

orders[order_datetime_cols] = orders[order_datetime_cols].apply(pd.to_datetime)
orders = orders.loc[orders['order_status'].eq('delivered')].copy()
orders = orders.dropna(
    subset=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ]
)

purchase_time = orders['order_purchase_timestamp']
estimated_delivery_time = orders['order_estimated_delivery_date']

orders['delay_days'] = (
    orders['order_delivered_customer_date'] - estimated_delivery_time
).dt.total_seconds().div(86400)
orders['purchase_month'] = purchase_time.dt.month
orders['purchase_dayofweek'] = purchase_time.dt.dayofweek
orders['purchase_hour'] = purchase_time.dt.hour
orders['is_weekend'] = purchase_time.dt.dayofweek.isin([5, 6]).astype(int)
orders['estimated_delivery_time_days'] = (
    estimated_delivery_time - purchase_time
).dt.total_seconds().div(86400)


In [ ]:
# This cell keeps the customer fields needed for V1 geography features.
customer_features_df = customers[[
    'customer_id',
    'customer_state',
    'customer_zip_code_prefix',
]].copy()


In [ ]:
# This cell joins item and product data and builds the V1 order-item aggregates.
product_feature_cols = [
    'product_id',
    'product_category_name',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
]

item_product_df = items.merge(
    products[product_feature_cols].copy(),
    on='product_id',
    how='left',
)
item_product_df['product_category_name'] = item_product_df['product_category_name'].fillna('unknown')
item_product_df['item_volume_cm3'] = (
    item_product_df['product_length_cm']
    * item_product_df['product_height_cm']
    * item_product_df['product_width_cm']
)

dominant_category_by_order = (
    item_product_df.groupby('order_id')['product_category_name']
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else 'unknown')
    .rename('dominant_category')
)

order_item_agg_df = (
    item_product_df.groupby('order_id', as_index=False)
    .agg(
        item_count=('order_item_id', 'count'),
        unique_products=('product_id', 'nunique'),
        unique_sellers=('seller_id', 'nunique'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        total_weight=('product_weight_g', lambda s: s.sum(min_count=1)),
        total_volume=('item_volume_cm3', lambda s: s.sum(min_count=1)),
        num_categories=('product_category_name', 'nunique'),
    )
    .merge(dominant_category_by_order.reset_index(), on='order_id', how='left')
)

In [ ]:
# This cell prepares seller geography fields and picks one seller per order for V1 geography features.
seller_features_df = sellers[[
    'seller_id',
    'seller_state',
    'seller_zip_code_prefix',
]].copy()

order_primary_seller_df = (
    items.groupby(['order_id', 'seller_id'], as_index=False)
    .agg(seller_item_count=('order_item_id', 'count'))
    .sort_values(['order_id', 'seller_item_count', 'seller_id'], ascending=[True, False, True])
    .drop_duplicates(subset='order_id')
    [['order_id', 'seller_id']]
    .rename(columns={'seller_id': 'primary_seller_id'})
)


In [ ]:
# This cell averages geolocation rows down to one ZIP-level lookup table.
geo_zip_df = (
    geolocation.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        zip_lat=('geolocation_lat', 'mean'),
        zip_lng=('geolocation_lng', 'mean'),
    )
)

In [ ]:
# This cell attaches ZIP-based coordinates to the V1 customer and seller feature tables.
customer_features_df = customer_features_df.merge(
    geo_zip_df.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'zip_lat': 'customer_lat',
        'zip_lng': 'customer_lng',
    }),
    on='customer_zip_code_prefix',
    how='left',
)
seller_features_df = seller_features_df.merge(
    geo_zip_df.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'zip_lat': 'seller_lat',
        'zip_lng': 'seller_lng',
    }),
    on='seller_zip_code_prefix',
    how='left',
)

In [ ]:
# This cell merges the V1 time, geography, and order-item features into the final model tables.
def haversine(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return earth_radius_km * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

final_model_df = orders[[
    'order_id',
    'customer_id',
    'delay_days',
    'purchase_month',
    'purchase_dayofweek',
    'purchase_hour',
    'is_weekend',
    'estimated_delivery_time_days',
]].copy()

final_model_df = final_model_df.merge(order_item_agg_df, on='order_id', how='left')
final_model_df = final_model_df.merge(customer_features_df, on='customer_id', how='left')
final_model_df = final_model_df.merge(order_primary_seller_df, on='order_id', how='left')
final_model_df = final_model_df.merge(
    seller_features_df.rename(columns={'seller_id': 'primary_seller_id'}),
    on='primary_seller_id',
    how='left',
)

final_model_df['same_state'] = (
    final_model_df['customer_state'] == final_model_df['seller_state']
).astype(int)
final_model_df['distance_km'] = haversine(
    final_model_df['customer_lat'],
    final_model_df['customer_lng'],
    final_model_df['seller_lat'],
    final_model_df['seller_lng'],
)

df = final_model_df.copy()
df_clean = final_model_df.drop(columns=[
    'order_id',
    'customer_id',
    'primary_seller_id',
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'customer_lat',
    'customer_lng',
    'seller_lat',
    'seller_lng',
]).copy()

In [14]:

# Negative carrier time — physically impossible
print("\ncarrier_time_hours < 0 (impossible):")
negative_carrier = df_clean[df_clean['carrier_time_hours'] < 0]
print(f"  Count: {len(negative_carrier)}")
print(f"  % of total: {len(negative_carrier)/len(df_clean):.2%}")

# Negative estimated delivery — also impossible
print("\nestimated_delivery_hours < 0 (impossible):")
negative_estimated = df_clean[df_clean['estimated_delivery_hours'] < 0]
print(f"  Count: {len(negative_estimated)}")
print(f"  % of total: {len(negative_estimated)/len(df_clean):.2%}")


carrier_time_hours < 0 (impossible):
  Count: 165
  % of total: 0.17%

estimated_delivery_hours < 0 (impossible):
  Count: 0
  % of total: 0.00%


In [15]:
# Drop negative carrier_time — physically impossible, timestamp bug in system
df = df[df['carrier_time_hours'] >= 0]

#drop null delay hours
df_clean = df_clean.dropna(subset=['delay_hours'])

In [16]:
print(f"shape:  {df_clean.shape}")
print(df_clean.info())
print(f"\nMissing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\ncolumns: {list(df_clean.columns)}")

shape:  (96470, 21)
<class 'pandas.DataFrame'>
Index: 96470 entries, 0 to 96477
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   delay_hours                   96470 non-null  float64
 1   approval_time_hours           96456 non-null  float64
 2   carrier_time_hours            96469 non-null  float64
 3   estimated_delivery_hours      96470 non-null  float64
 4   purchase_month                96470 non-null  int32  
 5   purchase_dayofweek            96470 non-null  int32  
 6   num_items                     96470 non-null  int64  
 7   total_price                   96470 non-null  float64
 8   total_freight                 96470 non-null  float64
 9   payment_installments          96469 non-null  float64
 10  payment_value                 96469 non-null  float64
 11  payment_type                  96469 non-null  str    
 12  payment_methods_used          96469 non-null  float64
 1

In [18]:
num_cols = df_clean.select_dtypes(include="number").columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
cat_cols = df_clean.select_dtypes(include="object").columns
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")
# Save training imputation values — reuse these on the test set to avoid leakage
train_num_medians = df_clean[num_cols].median()
train_cat_modes   = {col: df_clean[col].mode()[0] for col in cat_cols}

Shape after cleaning: (96470, 21)
Missing values remaining: 0


In [ ]:

# One-hot encode all categorical features
df_encoded = pd.get_dummies(df_clean, drop_first=False)

FEATURES = [c for c in df_encoded.columns if c not in ["delay_hours"]]
TARGET   = "delay_hours"

X = df_encoded[FEATURES]
y = df_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Features after one-hot encoding: {len(FEATURES)}")
print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows:     {X_test.shape[0]}")

Features after one-hot encoding: 95
Training rows: 77176
Test rows:     19294
